## 1. CFG and CNF

### CFG — Context-Free Grammar
A CFG describes how grammatical structures are formed.

```text
S  → NP VP
NP → Det N
VP → V NP
Det → 'the'
N → 'student'
V → 'reads'
```

### CNF — Chomsky Normal Form
For CKY, the important forms are:

```text
A → B C
A → 'word'
```

CKY uses `A → B C` to combine smaller spans.

## 2. CKY Parsing

**CKY (Cocke–Kasami–Younger)** is a dynamic-programming algorithm for parsing a CFG in CNF.

Basic idea:

```text
words → small phrases → larger phrases → complete sentence
```

### Solved example

Sentence:

> **the student reads books**

Grammar:

```text
S  → NP VP
NP → Det N
NP → N
VP → V NP
Det → 'the'
N → 'student' | 'books'
V → 'reads'
```

Reasoning:

```text
the → Det
student → N → NP
reads → V
books → N → NP

the student → NP
reads books → VP

NP + VP → S
```

Therefore, the sentence **can be parsed**.

In [ ]:
grammar = {
    "binary": [
        ("S", "NP", "VP"),
        ("NP", "Det", "N"),
        ("VP", "V", "NP")
    ],
    "lexical": {
        "the": ["Det"],
        "student": ["N"],
        "reads": ["V"],
        "books": ["N"]
    }
}

def cky_parse(words, grammar, start="S"):
    n = len(words)
    #Creates the CKY chart, where each cell stores the grammar categories
    chart = [[set() for _ in range(n + 1)] for _ in range(n)]

    # Find grammatical category of each word, put individual words into the chart
    for i, word in enumerate(words):
        chart[i][i+1].update(grammar["lexical"].get(word, []))
        if "N" in chart[i][i+1]:
            chart[i][i+1].add("NP")

    #Build larger phrases
    for length in range(2, n + 1):
        for i in range(n - length + 1):
            j = i + length

            #Try every possible split
            for k in range(i + 1, j):
                # Apply grammar rules
                for lhs, b, c in grammar["binary"]:
                    if b in chart[i][k] and c in chart[k][j]:
                        chart[i][j].add(lhs)

    return chart, start in chart[0][n]

sentence = "the student reads books".split()
chart, result = cky_parse(sentence, grammar)
print("Can be parsed?", result)

Can be parsed? True


### What does `chart[i][j]` mean?

It stores the grammatical categories that can generate the span from `i` to `j`.
For example:


```text
chart[0][2] = {NP}
```
means `the student` can form an `NP`.
The final check is:
```text
S ∈ chart[0][n]
```

## 3. NLTK Chart Parser

NLTK provides grammar and parsing tools:

```python
from nltk import CFG
from nltk.parse import ChartParser
```

`ChartParser` can generate parse trees from a CFG. We can use it to verify our CKY result.

In [ ]:
from nltk import CFG
from nltk.parse import ChartParser

nltk_grammar = CFG.fromstring("""
S -> NP VP
NP -> Det N
NP -> N
VP -> V NP
Det -> 'the'
N -> 'student' | 'books'
V -> 'reads'
""")

parser = ChartParser(nltk_grammar)
trees = list(parser.parse("the student reads books".split()))

print("Number of parse trees:", len(trees))
for tree in trees:
    print(tree)

Number of parse trees: 1
(S (NP (Det the) (N student)) (VP (V reads) (NP (N books))))


## 4. PCFG — Probabilistic CFG

A **PCFG** is a CFG where each grammar rule has a probability.

Example:

```text
S → NP VP [1.0]
NP → 'I' [0.1]
NP → Det N [0.3]
NP → NP PP [0.6]
VP → V NP [0.7]
VP → V NP PP [0.3]
```

### Why probabilities?

A sentence can have multiple valid parse trees.

Example:

> **I saw the girl with the camera.**

Possible interpretations:

1. The girl has the camera.
2. I used the camera to see the girl.

A CFG can find both structures. A PCFG assigns probabilities so we can compare them.

### Probability of a parse tree

Multiply the probabilities of all rules used:

```text
P(tree) = P(rule1) × P(rule2) × ... × P(ruleN)
```

# 5. Dependency Parsing with spaCy

### Sentence

> **The boy eats apples.**

We will use spaCy to find, for every word:
- its gramatical category tag,
- its head word,
- its dependency relation.


## Load spaCy

First, import spaCy and load its small English model.

In [ ]:
# If needed, install spaCy and the English model first:
# !pip install spacy
# !python -m spacy download en_core_web_sm

import spacy

nlp = spacy.load("en_core_web_sm")

## Parse the Sentence

Pass the sentence to the spaCy pipeline.

In [ ]:
sentence = "The boy eats apples."
doc = nlp(sentence)

print(sentence)

The boy eats apples.


## Examine Each Word

For each token, spaCy provides:

- `token.text` → the word
- `token.pos_` → gramatical category tag
- `token.head.text` → the word's syntactic head
- `token.dep_` → dependency relation

## Print Head → Dependent Relationships

Now we can directly display the dependency edges.

In [ ]:
for token in doc:
    if token.dep_ != "ROOT":
        print(f"{token.head.text} → {token.text} : {token.dep_}")

boy → The : det
eats → boy : nsubj
eats → apples : dobj
eats → . : punct


## Understanding the Output

For this sentence, the important relationships are expected to look roughly like:

```text
eats → boy       : nsubj
eats → apples   : obj
boy  → The      : det
```

The exact POS/dependency labels are produced by spaCy's pretrained English model.

## Find the Root

The token whose dependency relation is `ROOT` is the root of the dependency tree.

In [ ]:
for token in doc:
    if token.dep_ == "ROOT":
        print("Root word:", token.text)

Root word: eats


## Key Takeaway

spaCy automatically performs the dependency parsing for us.

```text
Sentence
   ↓
spaCy
   ↓
Dependency Parser
   ↓
Head + Dependent + Relation
```

This is the same basic idea students will use in the actual dependency-parsing assignment, but the assignment uses a different sentence and asks for additional output.


## 6. How the Topics Connect

- **CFG:** What structures are allowed?
- **CNF:** A form suitable for CKY.
- **CKY:** Can the sentence be parsed?
- **NLTK:** A ready-made parser for comparison.
- **PCFG:** Which valid parse is more probable?
- **Dependency Parsing:** Which words depend on which other words?